In [10]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
import numpy as np
import pandas as pd
import h5py
import os
from tqdm import tqdm 
from scipy import stats
from scipy.io import loadmat

/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_46684/1016450630.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [11]:
## helper functions for analysis 
#alignment with ezTrack location tracking data
def alignMiniscopeBehavCamTimestamps(savePath, sessionPath, behavCamFrameRate, miniscopeCamFrameRate):
    #'\\'.join(sessionPath.split(os.sep)[:-1])+'\\timestamp.dat'
    print(sessionPath.split(os.sep)[:-1])
    # load eZ track output and behavior camera timestamps from miniscope software 
    ezTrackOutput = pd.read_csv(sessionPath)
    timestampfile = pd.read_table('/'.join(sessionPath.split(os.sep)[:-1])+'/timeStamps.csv', delimiter=',')
    miniscope_timestampfile = pd.read_table('/'.join(sessionPath.split(os.sep)[:-1])+'/timeStampsMiniscope.csv', delimiter=',')
    
    timestampfile_td = timestampfile.set_index(pd.to_timedelta(np.linspace(0, (len(timestampfile)-1)*(1/behavCamFrameRate), len(timestampfile)), unit='s'), drop=False)
  
    miniscopetimestamp_td = miniscope_timestampfile.set_index(pd.to_timedelta(np.linspace(0, (len(miniscope_timestampfile)-1)*(1/miniscopeCamFrameRate), len(miniscope_timestampfile)), unit='s'), drop=False)
    
    behavCam_frames = []
    sys_clock_behavCam = []
    #create "key" for aligning miniscope frames to timestamp file
    #then create behavior TD and align
    for msCam_frame in tqdm(range(0, len(miniscopetimestamp_td['Frame Number']))):
        #get sys clock time of each miniscope recorded frame
        #sys_clock_msCam = time_stamps['sysClock'].loc[msCam_frame]
        #find behav cam frame closest to sys clock time of ms frame
        behavCam_frame = list(timestampfile_td.iloc[(timestampfile_td['Time Stamp (ms)']-miniscopetimestamp_td['Time Stamp (ms)'].iloc[msCam_frame]).abs().argsort()[:1]].index)[0]
        #this is the behavCamIndex that is closest to the corresponding miniscope frame 
        behavCam_frames.append(behavCam_frame)
        sys_clock_behavCam.append(timestampfile_td.loc[behavCam_frame]['Time Stamp (ms)'])

    behavCamIdxToAlign = [timestampfile_td.index.get_loc(idx) for idx in behavCam_frames]
    #ezTrackOutput

    miniscopetimestamp_td['closestBehavCamFrameIdx'] = behavCamIdxToAlign

    X_coor=[]
    Y_coor=[]
    Distance_px=[] 

    for i in miniscopetimestamp_td['closestBehavCamFrameIdx'].values:
        X_coor.append(ezTrackOutput.loc[i]['X'])
        Y_coor.append(ezTrackOutput.loc[i]['Y'])
        Distance_px.append(ezTrackOutput.loc[i]['Distance_px'])
    
    miniscopetimestamp_td['X_coor'] = X_coor
    miniscopetimestamp_td['Y_coor'] = Y_coor
    miniscopetimestamp_td['Distance_px'] = Distance_px
    
    return(miniscopetimestamp_td)

In [12]:
## load and do some preprocessing on the CNMFE traces 
def normalize(trace, percentile=True):
    """ Normalize a fluorescence trace by its max or its 99th percentile. """
    trace = trace - np.min(trace)
    if np.percentile(trace, 99) > 0:
        if percentile:
            trace = trace / np.percentile(trace, 99)
        else:
            trace = trace / np.max(trace)
    return trace

def load_and_filter_traces(extract_mat_path: str,
                           labels_mat_path: str,
                           label_key: str = 'labels_ex'
                          ) -> pd.DataFrame:
    """
    Load temporal_weights from extract_mat_path (v7.3) and one of the
    labels_* arrays from labels_mat_path, keep only columns where
    labels == 1, and return as a pandas DataFrame.

    Parameters
    ----------
    extract_mat_path : str
        Path to the v7.3 .mat file containing an 'output/temporal_weights' dataset.
    labels_mat_path : str
        Path to the .mat file (v7.3 or earlier) containing a 'labels' group/struct.
    label_key : str, optional
        Which labels field to use: one of 'labels_ex', 'labels_ml', or
        'labels_overall'. Default is 'labels_ex'.

    Returns
    -------
    pd.DataFrame
        Rows = frames, columns = kept cells (named 'cell_<original_index>').
    """
    # validate choice
    allowed = ('labels_ex','labels_ml','labels_overall')
    if label_key not in allowed:
        raise ValueError(f"label_key must be one of {allowed}, got '{label_key}'")

    # 1) load & transpose temporal_weights → shape (nFrames, nCells)
    with h5py.File(extract_mat_path, 'r') as f:
        tw = f['output']['temporal_weights'][()]  # often (nCells, nFrames)
    tw = np.asarray(tw).T

    # 2) try to load chosen labels via HDF5; if that fails, fall back to loadmat
    try:
        with h5py.File(labels_mat_path, 'r') as f:
            hl = f['labels'][label_key][()]
    except OSError:
        mat = loadmat(labels_mat_path,
                      struct_as_record=False,
                      squeeze_me=True)
        lbl = mat['labels']  # either a dict or a mat_struct
        # pull out the right attribute/key
        if isinstance(lbl, dict):
            hl = lbl[label_key]
        else:
            hl = getattr(lbl, label_key)

    # 3) shape‐check & squeeze
    hl = np.asarray(hl).squeeze()
    if tw.shape[1] != hl.size:
        raise ValueError(
            f"dimension mismatch: temporal_weights is {tw.shape}, "
            f"labels array '{label_key}' has length {hl.size}"
        )

    # 4) filter & build DataFrame
    mask    = (hl == 1)
    kept_i  = np.nonzero(mask)[0]
    filtered = tw[:, mask]
    cols    = [f'cell_{i}' for i in kept_i]

    return (pd.DataFrame(filtered, columns=cols), hl)

def print_h5_tree(name, obj):
    """
    Callback for h5py.File.visititems.
    Prints group/dataset name and, for datasets, its shape and dtype.
    """
    if isinstance(obj, h5py.Group):
        print(f"Group:   {name}/")
    elif isinstance(obj, h5py.Dataset):
        print(f"Dataset: {name}  — shape={obj.shape}, dtype={obj.dtype}")

    
def zScoreTraces(dirPath, CNMFE_real_cells):
    
    CNMFE_real_cells = CNMFE_real_cells.apply(pd.to_numeric, errors='coerce')
    C_normalized = CNMFE_real_cells.apply(lambda col: normalize(col), axis=0)
    C_z_scored = CNMFE_real_cells.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(CNMFE_real_cells)-1)*(1/20), len(CNMFE_real_cells)), unit='s'), drop=True)
    C_normalized_z_scored = C_normalized.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(C_normalized)-1)*(1/20), len(C_normalized)), unit='s'), drop=True)

    ##load spatial components by session
    # for v4 dimensions are 600x600 pixels
 
    C_normalized_z_scored.to_csv(dirPath+'_C_traces_filtered_origHz.csv')
    
    print('finished, saved:')
    print(dirPath+'_C_traces_filtered_origHz.csv')
    
    return(C_normalized_z_scored)

In [13]:
#calcium analysis data - extract and ActSort labels 
dirPath = r'/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/Part_4_Day1rec_DMSO_11/'
GCaMPData_EXTRACT_fName = r'1_37_motion_corrected.mat'
ActSortLabels_fName = r'Part4_Day1rec_11_precomputed_1_37_motion_corrected.mat'

#'labels_ex', 'labels_ml', or 'labels_overall'. Default is 'labels_ex'
GCAMP_traces_humanSorted, labels_human = load_and_filter_traces(dirPath+GCaMPData_EXTRACT_fName, dirPath+ActSortLabels_fName, 'labels_ex')
GCAMP_traces_modelSorted, labels_model = load_and_filter_traces(dirPath+GCaMPData_EXTRACT_fName, dirPath+ActSortLabels_fName, 'labels_ml')
GCAMP_traces_OvrallSorted, labels_overall = load_and_filter_traces(dirPath+GCaMPData_EXTRACT_fName, dirPath+ActSortLabels_fName, 'labels_overall')

GCAMP_traces_Zscore = zScoreTraces(dirPath, GCAMP_traces_OvrallSorted)

finished, saved:
/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/Part_4_Day1rec_DMSO_11/_C_traces_filtered_origHz.csv


In [14]:
#if you want to see where the human and model labels differ 
diff_inds = np.where(labels_human != labels_model)[0]
diff_inds

array([   2,    3,    4, ..., 1318, 1319, 1320])

In [15]:
#behavior analysis info
savePath = r'/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/Part_4_Day1rec_DMSO_11/'
sessionPath = r'/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/Part_4_Day1rec_DMSO_11/Mouse11Day1_custom_cropped_output_LocationOutput.csv'
behavCamFrameRate = 15
miniscopeCamFrameRate = 20 

behavCamDataAligned = alignMiniscopeBehavCamTimestamps(savePath, sessionPath, behavCamFrameRate, miniscopeCamFrameRate)

['', 'Users', 'johnmarshall', 'Documents', 'Analysis', 'miniscope_analysis', 'miniscopeLinearTrack', 'Part_4_Day1rec_DMSO_11']


100%|███████████████████████████████████| 36010/36010 [00:19<00:00, 1819.12it/s]


In [16]:
# align tracking data to CNMFE for the movies we've analyzed 
CNMFE_aligned = pd.concat([GCAMP_traces_Zscore, behavCamDataAligned.iloc[0:len(GCAMP_traces_Zscore)]], axis=1)
CNMFE_aligned.to_csv(dirPath+GCaMPData_EXTRACT_fName.strip(".mat")+'cellTracesAlignedToTracking.csv')

In [17]:
CNMFE_aligned

,cell_12,cell_13,cell_14,cell_15,cell_19,cell_21,cell_22,cell_25,cell_27,cell_29,...,cell_1314,cell_1315,cell_1317,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,X_coor,Y_coor,Distance_px
0 days 00:00:00,0.661929,0.413001,1.494183,-0.766036,-0.649710,1.740848,1.557957,1.099901,-0.516615,0.325547,...,-0.665207,1.701197,-0.900212,0,-21,0,0,1.000000,17.000000,0.000000
0 days 00:00:00.050000,0.755292,-0.171724,1.502397,-0.804833,-0.578913,0.702669,0.871224,0.749012,-1.251515,1.456305,...,-0.475032,0.931724,-1.672080,1,32,0,1,1.000000,18.961089,1.961089
0 days 00:00:00.100000,0.209402,-0.077480,1.217757,-1.435824,-0.455961,1.279007,1.067169,-0.128330,1.128362,0.105724,...,-0.578195,1.940201,-0.455581,2,80,0,2,1.000000,18.959677,0.001412
0 days 00:00:00.150000,0.526674,0.032346,1.614694,-0.098954,-0.607497,0.436783,1.197700,0.339635,-0.039178,0.438869,...,-0.047258,1.008860,-0.630109,3,131,0,2,1.000000,18.959677,0.001412
0 days 00:00:00.200000,0.459089,-0.321650,1.060506,-1.443203,-0.238626,1.110664,0.806883,0.256595,0.268090,0.245154,...,-0.109851,1.154502,-0.166875,4,182,0,3,1.000000,19.000000,0.040323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0 days 00:30:00.250000,-0.113662,-0.183885,2.704232,-0.686337,2.702309,-1.275401,0.425866,-0.983263,-0.419880,-0.099831,...,-0.177226,0.464474,-0.236355,36005,1824080,0,26150,14.353785,12.324779,0.300490
0 days 00:30:00.300000,0.496993,0.183803,2.817280,-0.086100,1.593364,-1.343665,0.255834,-0.948490,-1.401832,0.539685,...,-0.588805,0.165595,-0.015984,36006,1824131,0,26150,14.353785,12.324779,0.300490
0 days 00:30:00.350000,-0.718442,-0.011438,3.052339,-0.167741,2.722773,-1.151372,-0.057891,-0.856564,-0.928276,0.251885,...,-0.122325,1.053783,-0.669589,36007,1824181,0,26151,15.563084,12.077882,1.234246
0 days 00:30:00.400000,0.581232,0.216738,2.865067,0.430805,1.769982,-1.065549,0.112603,-0.886966,-0.139182,0.685274,...,0.157474,0.077088,0.367224,36008,1824232,0,26152,14.183544,12.114608,1.380029
